# Week 08 — Potential energy and conservation of energy

## Potansiyel enerji ve enerjinin korunumu

**Dates:** 9 November 2026 – 13 November 2026 · **Class:** 10 November 2026  
**Scope / Kapsam:** Gravitational/spring potential, conservative forces and energy accounting, including dissipative work.

**How to use this notebook:** read the physics and follow the worked algebra on paper — draw → choose a law → rearrange → substitute with units → check. Every problem has a step-by-step answer under *Answer and steps*; try the problem before opening it. The interactive graphs are optional checks: run the setup cells once, then any *Run the demonstration* cell, move a slider and read the status line under the controls. Python is not a learning requirement.

**TR:** Önce fiziği oku ve cebir adımlarını kâğıt üzerinde izle. Her problemin adım adım yanıtı *Answer and steps* altında gizlidir; önce kendin dene. Grafikler isteğe bağlı kontrollerdir: kurulum hücrelerini bir kez çalıştır, sonra gösterimi çalıştırıp kaydırıcıyı hareket ettir. Kod yazmak gerekmez.

## Contents / İçindekiler

1. [Before you start / Başlamadan önce](#w08-before)
   · [Setup for the interactive graphs (run once)](#w08-setup)
2. [Concepts, demonstrations and worked examples / Konular, gösterimler ve çözümlü örnekler](#w08-concepts) — 4 worked examples, 3 interactive graphs
3. [More worked examples from the question bank / Ek çözümlü örnekler](#w08-bank) — 5 examples
4. [Problem set with step-by-step answers / Problem seti](#w08-problems) — 6 problems
5. [Exit check and model responses / Çıkış kontrolü](#w08-exit)
   · [Solutions and next week / Çözümler ve gelecek hafta](#w08-next)

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)

## Lesson plan (3 hours) / Ders planı

<table width="100%">
<thead>
<tr>
<th align="left" width="80" scope="col">Minutes</th>
<th align="left" width="824" scope="col">Activity</th>
</tr>
</thead>
<tbody>
<tr>
<td>0–10</td>
<td>Retrieval: Restore $W_{\rm net}=K_f-K_i$ from Week 5. Distinguish an equation from its numerical substitution.</td>
</tr>
<tr>
<td>10–50</td>
<td>Potential energy and a consistent reference</td>
</tr>
<tr>
<td>50–60</td>
<td>Break / Ara</td>
</tr>
<tr>
<td>60–100</td>
<td>Conservation with springs and dissipative work</td>
</tr>
<tr>
<td>100–110</td>
<td>Break / Ara</td>
</tr>
<tr>
<td>110–150</td>
<td>Guided paper practice: core problems</td>
</tr>
<tr>
<td>150–170</td>
<td>Review difficult steps, one interactive check, questions</td>
</tr>
<tr>
<td>170–180</td>
<td>Exit check</td>
</tr>
</tbody>
</table>

The core route is enough for the lesson; intermediate and challenge problems and the demonstrations are for practice and review.

<a id="w08-before"></a>

## 1. Before you start / Başlamadan önce

**Retrieval question:** Restore $W_{\rm net}=K_f-K_i$ from Week 5. Distinguish an equation from its numerical substitution.

### Learning Objectives

By the end of this session you will be able to:

1. Define work done by a constant and a variable force
2. State and apply the work-energy theorem: $W_{\text{net}} = \Delta KE$
3. Calculate kinetic energy and relate it to speed changes
4. Define gravitational and elastic potential energy
5. Apply conservation of mechanical energy to solve problems
6. Compute instantaneous and average power
7. Build intuition through interactive energy diagrams and simulations

<a id="w08-setup"></a>

## Setup for the interactive graphs (run once) / Kurulum — bir kez çalıştır

Run the cells in this section once per session, then run any **Run the demonstration** cell below. Each demonstration shows a status line under its controls: **Updating…** while the graph is drawn, then the draw time and whether the sliders update live or on release. Nothing here needs to be edited.  
**TR:** Bu bölümdeki hücreleri oturum başına bir kez çalıştır; sonra istediğin gösterimi çalıştır. Kontrollerin altındaki durum satırı, grafiğin ne zaman güncellendiğini gösterir.

In [ ]:
#@title Run once — prepare the physics demonstrations
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (8, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('All imports successful.')

In [ ]:
#@title Run once — prepare the demonstration controls
"""Shared demonstration interface, embedded in every PHY101 notebook.

Only standard ipywidgets, IPython and matplotlib are used, so the notebooks stay
self-contained in Colab and in a local Jupyter. The design goals are:

* A slider change must always produce a visible reaction. The status line under
  the controls says "Updating…" immediately and reports the draw time afterwards.
* The graph is replaced through the same Output-widget route that
  ``ipywidgets.interact`` uses (``clear_output(wait=True)`` followed by a fresh
  display), which is the most widely tested path in Colab and Jupyter. No
  output-capturing context is used: ipykernel 7 dispatches widget messages
  concurrently and IPython's capture object breaks that dispatch.
* Live updates while dragging are switched on when a graph draws quickly and
  switched off (update on release) when it draws slowly, so the kernel never
  falls behind a fast slider.
"""
import functools
import sys
import time
import traceback

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import HTML, clear_output, display

# Force the inline backend. Otherwise a local kernel may choose a desktop
# backend and block at plt.show(), which looks like a frozen notebook.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

_physics_panels = []
_physics_callback_errors = []

# Draw-time thresholds (seconds) for switching live dragging on and off.
PHYSICS_LIVE_ON = 0.12
PHYSICS_LIVE_OFF = 0.25


def physics_frames(frame_count, maximum=60):
    """Sample display frames, retaining both endpoints and all simulation data."""
    count = int(frame_count)
    shown = min(count, maximum)
    if shown <= 1:
        return list(range(shown))
    return [round(index * (count - 1) / (shown - 1)) for index in range(shown)]


def physics_interval(frame_count, interval_ms):
    """Preserve first-to-last playback duration when display frames are sampled."""
    shown = len(physics_frames(frame_count))
    return interval_ms if shown <= 1 else interval_ms * (int(frame_count) - 1) / (shown - 1)


PHYSICS_STYLE = """<style>
.phy101-panel { border: 1px solid #a9b9c9; border-radius: 8px; padding: 8px; background: #fff; }
.phy101-controls { padding: 0 0 2px; box-sizing: border-box; }
.phy101-status { font-size: 12px; color: #4a5a6a; padding: 0 2px 6px; min-height: 18px; }
.phy101-status.busy { color: #b45309; }
.phy101-plot-output img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-plot-output .output_area { overflow: visible; }
.phy101-plot-output table { font-size: 13px; width: 100%; }
.phy101-plot-output .animation { max-width: 100%; }
.phy101-plot-output .animation img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-animation-panel { max-width: 100%; }
.phy101-animation-panel .animation { display: flex; flex-direction: column; }
.phy101-animation-panel .animation img { order: 2; max-width: 100%; height: auto; }
.phy101-animation-panel .anim-controls { order: 1; background: white; color: #172433; padding: 4px; }
@media (max-width: 650px) {
  .phy101-controls, .phy101-plot-output { width: 100% !important; }
}
</style>"""
display(HTML(PHYSICS_STYLE))


def physics_animation_html(animation):
    """A self-contained animation pane with playback controls kept in view."""
    return HTML(PHYSICS_STYLE + '<div class="phy101-animation-panel">' +
                animation.to_jshtml(default_mode="once") + "</div>")


def _physics_controls(items):
    """Lay the controls out as a wrapping toolbar with full-length labels."""
    flat = []
    for item in items:
        if isinstance(item, (widgets.HBox, widgets.VBox)):
            flat.extend(item.children)
        else:
            flat.append(item)
    for control in flat:
        if hasattr(control, "style") and "description_width" in control.style.traits():
            control.style.description_width = "initial"
        control.layout.width = "310px"
        control.layout.flex = "0 1 310px"
        control.layout.max_width = "100%"
        control.layout.min_width = "0"
        control.layout.margin = "2px 6px 2px 0"
        if isinstance(control, widgets.Button):
            control.layout.width = "auto"
            control.layout.flex = "0 0 auto"
    # Repeat the style inside the widget tree: Colab isolates output frames.
    style = widgets.HTML(value=PHYSICS_STYLE, layout=widgets.Layout(display="none"))
    box = widgets.Box([style] + flat, layout=widgets.Layout(
        display="flex", flex_flow="row wrap", align_items="center",
        width="100%", min_width="0", max_width="100%"))
    box.add_class("phy101-controls")
    return box


def _physics_output(output):
    output.layout = widgets.Layout(
        width="100%", min_width="0", max_width="100%",
        height="auto", overflow="visible", margin="0")
    output.add_class("phy101-plot-output")
    return output


def physics_panel(controls, output):
    """Button-driven demos: a compact control toolbar directly above the result."""
    panel = widgets.Box([_physics_controls(controls), _physics_output(output)],
        layout=widgets.Layout(display="flex", flex_flow="column",
                              align_items="stretch", width="100%"))
    panel.add_class("phy101-panel")
    return panel


def physics_show_figure(figure):
    """Display one inline figure and close its pyplot registration afterwards."""
    import matplotlib.pyplot as plt
    display(figure)
    plt.close(figure)


def physics_vector_axes(axes, points):
    """Equal x/y scales and limits covering all arrow endpoints, including sums."""
    import numpy as np
    coordinates = np.asarray(points, dtype=float).reshape(-1, 2)
    span = max(1.0, float(np.max(np.abs(coordinates)))) * 1.22
    axes.set(xlim=(-span, span), ylim=(-span, span), xlabel="x component", ylabel="y component")
    axes.set_aspect("equal", adjustable="box")
    axes.axhline(0, color="#718096", linewidth=0.7)
    axes.axvline(0, color="#718096", linewidth=0.7)
    axes.grid(alpha=0.2)


class PhysicsPanel:
    """Controls, a status line and one Output widget that shows the latest result."""

    def __init__(self, function, controls):
        self.f = function
        self.controls = controls
        self.out = _physics_output(widgets.Output())
        self.status = widgets.HTML(value="")
        self.status.add_class("phy101-status")
        self.seconds = None
        self.live = True
        self.updates = 0
        self.last_outputs = 0
        self.figures = 0
        self.error = None
        visible = []
        for control in controls.values():
            if isinstance(control, widgets.fixed):
                continue
            visible.append(control)
            if hasattr(control, "continuous_update"):
                control.continuous_update = True
            control.observe(self._changed, names="value")
        self.widget = widgets.VBox([_physics_controls(visible), self.status, self.out],
                                   layout=widgets.Layout(width="100%"))
        self.widget.add_class("phy101-panel")
        self.children = self.widget.children
        _physics_panels.append(self)
        self.render()

    # Compatibility with the earlier validation code.
    @property
    def layout(self):
        return self.widget.layout

    def _changed(self, change):
        self.render()

    def _set_live(self, live):
        if live == self.live:
            return
        self.live = live
        for control in self.controls.values():
            if hasattr(control, "continuous_update"):
                control.continuous_update = live

    def render(self):
        self.status.value = "⏳ Updating… / Güncelleniyor…"
        self.status.add_class("busy")
        started = time.perf_counter()
        kwargs = {name: control.value for name, control in self.controls.items()}
        self.error = None
        self.figures = 0
        # Count everything the demonstration shows (figures, HTML, animations, text)
        # by wrapping the display publisher and stdout for this draw only.
        shell = get_ipython()
        publisher = getattr(shell, "display_pub", None) if shell is not None else None
        if publisher is not None:
            original_publish = publisher.publish

            def counting_publish(*args, **kwargs):
                self.figures += 1
                return original_publish(*args, **kwargs)
            publisher.publish = counting_publish
        stdout = sys.stdout
        original_write = stdout.write

        def counting_write(text):
            if text.strip():
                self.figures += 1
            return original_write(text)
        stdout.write = counting_write
        try:
            # The previous result stays visible until the new one arrives.
            with self.out:
                clear_output(wait=True)
                try:
                    result = self.f(**kwargs)
                    from ipywidgets.widgets.interaction import show_inline_matplotlib_plots
                    show_inline_matplotlib_plots()
                    if result is not None:
                        display(result)
                except Exception:
                    self.error = traceback.format_exc()
                    _physics_callback_errors.append((getattr(self.f, "__name__", "callback"), self.error))
                    print(self.error)
        finally:
            if publisher is not None and publisher.__dict__.get("publish") is counting_publish:
                del publisher.publish
            if stdout.__dict__.get("write") is counting_write:
                del stdout.write
        self.last_outputs = self.figures
        self.seconds = time.perf_counter() - started
        self.updates += 1
        if self.seconds > PHYSICS_LIVE_OFF:
            self._set_live(False)
        elif self.seconds < PHYSICS_LIVE_ON:
            self._set_live(True)
        self.status.remove_class("busy")
        mode = ("updates while you drag / sürüklerken güncellenir" if self.live
                else "updates when you release the slider / kaydırıcıyı bırakınca güncellenir")
        self.status.value = (f"✓ Drawn in {self.seconds:.2f} s · {mode}" if self.error is None
                             else "⚠ The demonstration reported an error; see the message below.")


def physics_interactive(function, **controls):
    """Build a panel like ipywidgets.interactive, returning the panel object."""
    @functools.wraps(function)
    def checked(*args, **kwargs):
        return function(*args, **kwargs)

    return PhysicsPanel(checked, controls)


def physics_interact(function=None, **controls):
    """Support both @physics_interact(...) and physics_interact(function, ...)."""
    if function is None:
        return lambda function: physics_interact(function, **controls)
    panel = physics_interactive(function, **controls)
    function.widget = panel.widget
    function.panel = panel
    display(panel.widget)
    return function


<a id="w08-concepts"></a>

## 2. Concepts, demonstrations and worked examples / Konular, gösterimler ve çözümlü örnekler

**Before you move a slider:** Predict kinetic and gravitational energy at launch, the peak and return. Explain why total mechanical energy stays constant in this model.

### After the midterm: put gravitational work into an energy account

Recall $W_{\rm net}=K_f-K_i$. For a vertical displacement with upward positive, gravity does
$$W_g=-mg(y_f-y_i)=mgy_i-mgy_f.$$
Define gravitational potential energy $U_g=mgy$ relative to one chosen zero height. Then $W_g=U_{g,i}-U_{g,f}$. Substitute into work–energy and add $U_{g,f}$ to both sides:
$$K_i+U_{g,i}=K_f+U_{g,f}.$$
We have reorganized the same physics. Do not count gravitational work a second time after including $U_g$.

A spring stores $U_s=\tfrac12kx^2$, with $x$ measured from its natural length. This is the triangular area under the applied-force-versus-extension graph. If other forces such as friction do work,
$$K_i+U_i+W_{\rm other}=K_f+U_f.$$
Friction usually makes $W_{\rm other}<0$ for the chosen moving system. Mechanical energy decreases while energy transfers to thermal/internal forms.

**TR:** Önce sıfır yüksekliği seç. $mgh$ yalnızca seçtiğin referansa göre anlamlıdır; iki durum arasında aynı referansı koru.

### Conservation of mechanical energy / Mekanik enerjinin korunumu

When only conservative forces (gravity, springs) do work:

$$KE_i + PE_i = KE_f + PE_f$$

$$E_{\text{mech}} = KE + PE = \text{constant}$$

**Read a complete energy account / Enerji bilançosunu oku.** A $2.0\,\mathrm{kg}$ block starts from rest $1.0\,\mathrm m$ above the chosen zero. Compare the same descent with and without $4.0\,\mathrm J$ of frictional work:

<table width="100%">
<thead>
<tr>
<th align="left" width="128" scope="col">Case</th>
<th align="left" width="88" scope="col">$K_i+U_i$</th>
<th align="left" width="80" scope="col">$W_{\rm other}$</th>
<th align="left" width="104" scope="col">$K_f$ at $h_f=0$</th>
<th align="left" width="200" scope="col">$v_f$</th>
</tr>
</thead>
<tbody>
<tr>
<td>Frictionless</td>
<td>$19.62\,\mathrm J$</td>
<td>$0\,\mathrm J$</td>
<td>$19.62\,\mathrm J$</td>
<td>$\sqrt{2(19.62)/2.0}=4.43\,\mathrm{m/s}$</td>
</tr>
<tr>
<td>With friction</td>
<td>$19.62\,\mathrm J$</td>
<td>$-4.0\,\mathrm J$</td>
<td>$15.62\,\mathrm J$</td>
<td>$3.95\,\mathrm{m/s}$</td>
</tr>
</tbody>
</table>

Both follow $K_f=K_i+U_i+W_{\rm other}-U_f$. **TR:** Sürtünmenin eksi işareti enerji denklemine zaten girdi; aynı kaybı tekrar çıkarmayın. Tablodaki her enerji sütununun birimi joule'dür.

#### ⏱️ Checkpoint 1 of 1 — Think · Pair · Explain

A horizontal frictionless spring has $k=100\,\mathrm{N/m}$, compression $x=0.20\,\mathrm m$, and launches a $0.50\,\mathrm{kg}$ block. **Think:** predict how doubling compression changes speed. **Pair:** solve the energy equation for speed before substituting. **Explain:** compare $x=0.20$ and $0.40\,\mathrm m$.

**TR:** Sıkışma iki katına çıkınca enerji ile sürat aynı oranda mı değişir?

Write your prediction first, then compare with the model below. Include the governing law, units and one sense check; use the pause to raise any remaining question.

<details><summary>Model answer / Örnek yanıt — open after trying</summary>

$$\frac12kx^2=\frac12mv^2\quad\Rightarrow\quad v^2=\frac{k}{m}x^2\quad\Rightarrow\quad v=x\sqrt{\frac{k}{m}}.$$

<table width="100%">
<thead>
<tr>
<th align="left" width="112" scope="col">Compression</th>
<th align="left" width="200" scope="col">Spring energy</th>
<th align="left" width="192" scope="col">Speed</th>
</tr>
</thead>
<tbody>
<tr>
<td>$0.20\,\mathrm m$</td>
<td>$\frac12(100)(0.20)^2=2.0\,\mathrm J$</td>
<td>$\sqrt{2(2.0)/0.50}=2.83\,\mathrm{m/s}$</td>
</tr>
<tr>
<td>$0.40\,\mathrm m$</td>
<td>$8.0\,\mathrm J$</td>
<td>$5.66\,\mathrm{m/s}$</td>
</tr>
</tbody>
</table>

Energy quadruples but speed doubles. **TR:** Enerjide hem sıkışmanın hem süratin karesi bulunduğundan, sürat sıkışmayla doğru orantılıdır.

**Your explanation / Açıklaman:** Compare your original prediction with this model and name one step you would change.

</details>

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Interactive: Energy Bar Chart

An object is launched upward with initial speed $v_0$. Watch how kinetic energy converts to potential energy and back. The total mechanical energy stays constant (no air resistance).

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def energy_bar_chart(v0=15.0, mass=2.0):
    """Animate energy bar chart for vertical throw."""
    g = 9.81
    h_max = v0**2 / (2 * g)
    t_total = 2 * v0 / g
    n_frames = 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6),
                                    gridspec_kw={'width_ratios': [1, 1.2]})

    # Left: trajectory
    ax1.set_xlim(-1, 1)
    ax1.set_ylim(-0.5, h_max * 1.2)
    ax1.set_ylabel('Height (m)')
    ax1.set_title('Trajectory', fontsize=13)
    ax1.get_xaxis().set_visible(False)
    ax1.axhline(y=0, color='brown', lw=3)  # ground

    ball, = ax1.plot([], [], 'ro', markersize=16)
    height_text = ax1.text(0.3, 0, '', fontsize=10)

    # Right: bar chart
    E_total = 0.5 * mass * v0**2
    ax2.set_xlim(-0.5, 3.5)
    ax2.set_ylim(0, E_total * 1.3)
    ax2.set_title('Energy Bar Chart', fontsize=13)
    ax2.set_ylabel('Energy (J)')
    ax2.set_xticks([0, 1, 2])
    ax2.set_xticklabels(['KE', 'PE', 'Total E'])

    bar_ke = ax2.bar(0, 0, width=0.6, color='dodgerblue', edgecolor='black', label='KE')
    bar_pe = ax2.bar(1, 0, width=0.6, color='orange', edgecolor='black', label='PE')
    bar_te = ax2.bar(2, 0, width=0.6, color='green', edgecolor='black', label='Total')
    ax2.axhline(y=E_total, color='green', ls='--', alpha=0.5)
    ax2.legend(fontsize=10)

    ke_text = ax2.text(0, 0, '', ha='center', fontsize=9)
    pe_text = ax2.text(1, 0, '', ha='center', fontsize=9)
    te_text = ax2.text(2, 0, '', ha='center', fontsize=9)
    info = ax1.text(-0.9, h_max*1.1, '', fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    def init():
        ball.set_data([], [])
        return ball,

    def update(frame):
        t = t_total * frame / n_frames
        v = v0 - g * t
        h = v0 * t - 0.5 * g * t**2
        h = max(h, 0)

        KE = 0.5 * mass * v**2
        PE = mass * g * h

        ball.set_data([0], [h])
        height_text.set_position((0.3, h))
        height_text.set_text(f'h={h:.1f} m\nv={abs(v):.1f} m/s')

        bar_ke[0].set_height(KE)
        bar_pe[0].set_height(PE)
        bar_te[0].set_height(KE + PE)

        ke_text.set_position((0, KE + E_total*0.02))
        ke_text.set_text(f'{KE:.0f} J')
        pe_text.set_position((1, PE + E_total*0.02))
        pe_text.set_text(f'{PE:.0f} J')
        te_text.set_position((2, (KE+PE) + E_total*0.02))
        te_text.set_text(f'{KE+PE:.0f} J')

        info.set_text(
            f'mass = {mass:.1f} kg\n'
            f'v0 = {v0:.1f} m/s\n'
            f'h_max = {h_max:.1f} m\n'
            f'E_total = {E_total:.0f} J'
        )

        return ball,

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=physics_frames(n_frames), interval=physics_interval(n_frames, 50), blit=False)
    plt.close(fig)
    return physics_animation_html(anim)

physics_interact(energy_bar_chart,
         v0=FloatSlider(min=5, max=30, step=1, value=15, description='v0 (m/s)'),
         mass=FloatSlider(min=0.5, max=10, step=0.5, value=2.0, description='Mass (kg)'));

### Interactive: Spring-Block Oscillation with Energy

A block attached to a spring oscillates back and forth. Watch the continuous exchange between kinetic energy and elastic potential energy.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def spring_block_energy(k=20.0, mass=1.0, A=0.5):
    """Animate spring-block oscillation with energy visualization."""
    omega = np.sqrt(k / mass)
    T = 2 * np.pi / omega
    E_total = 0.5 * k * A**2
    n_frames = 120

    fig = plt.figure(figsize=(13, 8))
    ax1 = fig.add_subplot(2, 1, 1)  # spring-block
    ax2 = fig.add_subplot(2, 2, 3)  # energy vs time
    ax3 = fig.add_subplot(2, 2, 4)  # bar chart

    # Top: spring-block diagram
    ax1.set_xlim(-1.5, 1.5)
    ax1.set_ylim(-0.5, 0.5)
    ax1.set_title('Spring-Block System', fontsize=13)
    ax1.set_xlabel('Position (m)')
    ax1.axvline(x=0, color='gray', ls=':', alpha=0.5, label='Equilibrium')
    ax1.get_yaxis().set_visible(False)

    # Wall
    ax1.plot([-1.5, -1.5], [-0.3, 0.3], 'k-', lw=3)
    for yy in np.linspace(-0.3, 0.3, 6):
        ax1.plot([-1.5, -1.6], [yy, yy-0.05], 'k-', lw=1)

    block = plt.Rectangle((0, -0.15), 0.2, 0.3, fc='steelblue', ec='black', lw=2)
    ax1.add_patch(block)
    spring_line, = ax1.plot([], [], 'r-', lw=2)
    pos_text = ax1.text(0, 0.35, '', fontsize=10, ha='center')

    # Bottom-left: energy vs time
    t_arr = np.linspace(0, 2*T, 500)
    x_arr = A * np.cos(omega * t_arr)
    KE_arr = 0.5 * mass * (A * omega * np.sin(omega * t_arr))**2
    PE_arr = 0.5 * k * x_arr**2

    ax2.plot(t_arr, KE_arr, 'b-', label='KE', alpha=0.5)
    ax2.plot(t_arr, PE_arr, 'orange', label='PE', alpha=0.5)
    ax2.axhline(y=E_total, color='green', ls='--', alpha=0.4, label='Total')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Energy (J)')
    ax2.set_title('Energy vs Time', fontsize=12)
    ax2.legend(fontsize=9)
    time_marker, = ax2.plot([], [], 'ko', markersize=8)

    # Bottom-right: bar chart
    ax3.set_xlim(-0.5, 2.5)
    ax3.set_ylim(0, E_total * 1.4)
    ax3.set_title('Energy Bar Chart', fontsize=12)
    ax3.set_ylabel('Energy (J)')
    ax3.set_xticks([0, 1, 2])
    ax3.set_xticklabels(['KE', 'PE_s', 'Total'])
    bar_ke = ax3.bar(0, 0, width=0.5, color='dodgerblue', ec='black')
    bar_pe = ax3.bar(1, 0, width=0.5, color='orange', ec='black')
    bar_te = ax3.bar(2, E_total, width=0.5, color='green', ec='black', alpha=0.5)

    def draw_spring(x_start, x_end, n_coils=12):
        """Generate spring coordinates."""
        length = x_end - x_start
        t_spring = np.linspace(0, 1, n_coils * 20)
        xs = x_start + length * t_spring
        ys = 0.08 * np.sin(2 * np.pi * n_coils * t_spring)
        # Flatten ends
        ys[:10] *= np.linspace(0, 1, 10)
        ys[-10:] *= np.linspace(1, 0, 10)
        return xs, ys

    def init():
        return block,

    def update(frame):
        t = 2 * T * frame / n_frames
        x = A * np.cos(omega * t)
        v = -A * omega * np.sin(omega * t)

        KE = 0.5 * mass * v**2
        PE = 0.5 * k * x**2

        # Update block position
        block.set_x(x - 0.1)

        # Update spring
        sx, sy = draw_spring(-1.5, x - 0.1)
        spring_line.set_data(sx, sy)

        pos_text.set_position((x, 0.35))
        pos_text.set_text(f'x={x:.2f} m, v={v:.2f} m/s')

        # Time marker
        time_marker.set_data([t], [KE])

        # Bar chart
        bar_ke[0].set_height(KE)
        bar_pe[0].set_height(PE)

        return block, spring_line

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=physics_frames(n_frames), interval=physics_interval(n_frames, 50), blit=False)
    plt.tight_layout()
    plt.close(fig)
    return physics_animation_html(anim)

physics_interact(spring_block_energy,
         k=FloatSlider(min=5, max=100, step=5, value=20, description='k (N/m)'),
         mass=FloatSlider(min=0.5, max=5, step=0.5, value=1.0, description='Mass (kg)'),
         A=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.5, description='Amp (m)'));

### Interactive: Roller Coaster Energy Simulator

A ball rolls along a track with hills and valleys. Watch how energy converts between KE and PE. The track profile is defined by a function, and the ball follows it under gravity (frictionless).

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def roller_coaster(v0=2.0, track_type=1):
    """Roller coaster energy simulator."""
    g = 9.81
    mass = 1.0

    # Track profiles
    x_track = np.linspace(0, 10, 500)
    if track_type == 1:
        # Hills and valleys
        y_track = 2.0 + 1.2*np.sin(0.8*x_track) + 0.5*np.cos(1.6*x_track)
        title = 'Track: Hills and Valleys'
    elif track_type == 2:
        # Big drop then loop shape
        y_track = 3.0 - 0.3*x_track + 0.8*np.sin(1.2*x_track)
        y_track = np.maximum(y_track, 0.2)
        title = 'Track: Big Drop'
    else:
        # Wavy with increasing amplitude
        y_track = 2.0 + 0.15*x_track*np.sin(1.0*x_track)
        y_track = np.maximum(y_track, 0.2)
        title = 'Track: Increasing Waves'

    h0 = y_track[0]
    E_total = 0.5 * mass * v0**2 + mass * g * h0

    # Precompute ball position along track
    # Ball can only go where KE >= 0
    positions = []
    KEs = []
    PEs = []
    for i, (xi, yi) in enumerate(zip(x_track, y_track)):
        PE = mass * g * yi
        KE = E_total - PE
        if KE < 0:
            break  # ball cannot reach here
        positions.append((xi, yi))
        KEs.append(KE)
        PEs.append(PE)

    n_frames = len(positions)
    if n_frames < 5:
        print('Initial speed too low -- ball cannot move along the track.')
        return

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

    # Top: track with ball
    ax1.plot(x_track, y_track, 'k-', lw=3)
    ax1.fill_between(x_track, 0, y_track, alpha=0.1, color='brown')
    ax1.set_xlim(-0.5, 11)
    ax1.set_ylim(-0.5, max(y_track)*1.5)
    ax1.set_title(title, fontsize=13)
    ax1.set_xlabel('x (m)')
    ax1.set_ylabel('Height (m)')

    ball, = ax1.plot([], [], 'ro', markersize=14, zorder=5)
    # Energy reference line
    ax1.axhline(y=E_total/(mass*g), color='green', ls='--', alpha=0.4, label='Max reachable height')
    ax1.legend(fontsize=9)

    info = ax1.text(7, max(y_track)*1.3, '', fontsize=10,
                    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    # Bottom: stacked energy area
    x_pos = [p[0] for p in positions]
    ax2.fill_between(x_pos, 0, KEs, alpha=0.6, color='dodgerblue', label='KE')
    ax2.fill_between(x_pos, KEs, [k+p for k,p in zip(KEs, PEs)], alpha=0.6, color='orange', label='PE')
    ax2.set_xlabel('x (m)')
    ax2.set_ylabel('Energy (J)')
    ax2.set_title('Energy along the track', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.set_xlim(-0.5, 11)

    energy_marker, = ax2.plot([], [], 'k|', markersize=20, markeredgewidth=3)

    def init():
        ball.set_data([], [])
        return ball,

    def update(frame):
        idx = frame % n_frames
        xi, yi = positions[idx]
        ke = KEs[idx]
        pe = PEs[idx]
        v = np.sqrt(2 * ke / mass)

        ball.set_data([xi], [yi])
        energy_marker.set_data([xi], [ke])

        info.set_text(
            f'v = {v:.2f} m/s\n'
            f'h = {yi:.2f} m\n'
            f'KE = {ke:.1f} J\n'
            f'PE = {pe:.1f} J\n'
            f'E_total = {ke+pe:.1f} J'
        )
        return ball,

    anim = FuncAnimation(fig, update, init_func=init,
                         frames=physics_frames(n_frames), interval=physics_interval(n_frames, 30), blit=False)
    plt.tight_layout()
    plt.close(fig)
    return physics_animation_html(anim)

physics_interact(roller_coaster,
         v0=FloatSlider(min=0, max=8, step=0.5, value=2.0, description='v0 (m/s)'),
         track_type=IntSlider(min=1, max=3, step=1, value=1, description='Track'));

### Worked examples / Çözümlü örnekler

#### Example 2 -- Spring launcher

A spring ($k=500\,\mathrm{N/m}$) is compressed by $0.2\,\mathrm m$ and launches a $0.1\,\mathrm{kg}$ ball vertically. How high does the ball go?

**Paper solution / Kâğıt üzerinde çözüm.** Take the ball's initially compressed position as height zero, neglect spring mass and drag, and let the ball start and finish at rest. The spring initially stores $U_s=\tfrac12(500)(0.20)^2=10\,\mathrm J$. At the highest point all of this is gravitational energy:
$$10=(0.10)(9.81)h=0.981h.$$
Divide both sides by $0.981\,\mathrm N$: **$h=10/0.981=10.19\,\mathrm m$ above the initial compressed position**.

The reference point matters. If the ball separates when the vertical spring reaches its natural length, it has already risen $0.20\,\mathrm m$. Its kinetic energy at separation is $10-mg(0.20)=9.8038\,\mathrm J$, giving $v=\sqrt{2(9.8038)/0.10}=14.00\,\mathrm{m/s}$. It rises another $10.19-0.20=9.99\,\mathrm m$ above separation. A calculation that uses all $10\,\mathrm J$ as launch kinetic energy ignores the small gravitational gain during spring expansion.

**Türkçe:** “Ne kadar yükselir?” sorusunda yüksekliğin nereden ölçüldüğünü mutlaka yaz. Yay açılırken de top yukarı çıkar ve yerçekimi potansiyel enerjisi artar.

In [ ]:
#@title Optional numerical check — the algebra is explained above
k, x_comp, m = 500, 0.2, 0.1
g = 9.81
PE_spring = 0.5 * k * x_comp**2
# Gravity takes energy while the block rises through the spring compression.
KE_at_separation = PE_spring - m * g * x_comp
v_launch = np.sqrt(2 * KE_at_separation / m)
h_above_initial = PE_spring / (m * g)
h_above_separation = h_above_initial - x_comp
print(f'Stored spring energy: {PE_spring:.2f} J')
print(f'Speed when the spring reaches natural length: {v_launch:.3f} m/s')
print(f'Maximum rise above the initial compressed position: {h_above_initial:.3f} m')
print(f'Further rise after separation: {h_above_separation:.3f} m')
assert np.isclose(0.5 * m * v_launch**2 + m * g * x_comp, PE_spring)


#### Example 3 -- Block on incline with friction

A $4\,\mathrm{kg}$ block starts at the top of a $2\,\mathrm m$ high incline (angle $25^\circ$) with initial speed $1\,\mathrm{m/s}$. The kinetic friction coefficient is 0.15. What is the speed at the bottom?

**Work it by stages / Aşama aşama çözüm.** Use the bottom as $U_g=0$. The friction force acts along the ramp, so its work uses ramp length, not vertical height:

$$\begin{aligned}s&=\frac{h}{\sin25^\circ}=\frac{2.0}{\sin25^\circ}=4.7324\,\mathrm m,\\N&=mg\cos25^\circ=35.5635\,\mathrm N,\\W_f&=-\mu_kNs=-(0.15)(35.5635)(4.7324)=-25.2451\,\mathrm J.\end{aligned}$$

$$\begin{aligned}\frac12mv_f^2&=\frac12mv_i^2+mgh+W_f\\&=2.0+78.48-25.2451=55.2349\,\mathrm J,\\v_f^2&=\frac{2(55.2349\,\mathrm J)}{4.0\,\mathrm{kg}}=27.6174\,\mathrm{m^2/s^2},\\v_f&=\boxed{5.26\,\mathrm{m/s}}.\end{aligned}$$

Friction lowers the result below the frictionless $\sqrt{1^2+2(9.81)(2)}=6.34\,\mathrm{m/s}$. **TR:** Önce normal kuvvet, sonra sürtünme kuvveti, sonra sürtünme işi bulunur. Yatay olmayan yolda $N=mg$ yazmak doğru değildir.

In [ ]:
#@title Optional numerical check — the algebra is explained above
m, h, v_i, mu_k, theta_deg = 4, 2, 1, 0.15, 25
g = 9.81
theta = np.radians(theta_deg)
d = h / np.sin(theta)  # distance along incline

# Friction force
N = m * g * np.cos(theta)
f_k = mu_k * N
W_friction = -f_k * d  # negative (opposes motion)

# Energy: KE_i + PE_i + W_friction = KE_f
KE_i = 0.5 * m * v_i**2
PE_i = m * g * h
KE_f = KE_i + PE_i + W_friction
v_f = np.sqrt(2 * KE_f / m)

print(f'Distance along incline: {d:.2f} m')
print(f'Work by friction: {W_friction:.2f} J')
print(f'KE_initial + PE_initial = {KE_i + PE_i:.2f} J')
print(f'KE_final = {KE_f:.2f} J')
print(f'Speed at bottom: {v_f:.2f} m/s')

### Worked example: use the actual ramp length for friction

Module 06's example has a 4 kg block descending a 2 m high, 25° ramp with initial speed 1 m/s and $\mu_k=0.15$. Choose the bottom as zero height; the block is already sliding down.

Ramp distance is $s=h/\sin25^\circ=4.7324$ m. The normal force is $N=mg\cos25^\circ=35.5635$ N, so $f_k=0.15N=5.3345$ N and $W_f=-f_ks=-25.2451$ J.
$$K_i=\tfrac12(4)(1^2)=2\,\mathrm J,\quad U_i=4(9.81)(2)=78.48\,\mathrm J.$$
$$K_f=K_i+U_i+W_f=2+78.48-25.2451=55.2349\,\mathrm J.$$
$$v_f=\sqrt{2K_f/m}=\sqrt{2(55.2349)/4}=5.2552\,\mathrm{m/s}.$$
**Check:** Without friction, $v_f=\sqrt{1^2+2g(2)}=6.3435$ m/s, so the smaller result is reasonable. Use full precision in intermediate calculations.

**TR:** Sürtünme eğik düzlem boyunca etkir; yaptığı işte düşey yüksekliği değil, eğik yol uzunluğunu kullan.

<a id="w08-bank"></a>

## 3. More worked examples from the question bank / Soru bankasından ek çözümlü örnekler

Each example gives the complete reasoning: the law, the rearrangement, the substitution with units, the physical check and a Turkish note on the difficult step. Cover the next line and try to produce it yourself.

<a id="qb07-01"></a>

#### A swing: convert angle into vertical height first

**Problem.** You do $174\,\mathrm J$ of work pulling a rider from the lowest position of a swing to rest with its chain $32.0^\circ$ from vertical. The chain is $5.10\,\mathrm m$ long. Find the rider's mass, neglecting friction and the seat's mass. Use $g=9.80\,\mathrm{m/s^2}$.

**Find the height change from geometry.** The angle is measured from vertical. Initially the rider is a distance $L$ below the pivot; finally the vertical distance below the pivot is $L\cos\theta$. Subtracting these distances gives the rise:

$$\Delta h=L-L\cos\theta=L(1-\cos\theta).$$

This is a vertical height, not the arc length along the swing's curved path. Gravitational potential energy depends on the height change. Numerically,

$$\Delta h=(5.10\,\mathrm m)(1-\cos32.0^\circ)=0.77495\,\mathrm m.$$

**Use the energy balance.** The rider starts and ends at rest, so $\Delta K=0$. With negligible friction, your external work increases gravitational potential energy:

$$W=\Delta U_g=mg\Delta h.$$

Isolate the mass by dividing both sides by $g\Delta h$:

$$\begin{aligned}
m&=\frac{W}{g\Delta h}
=\frac{W}{gL(1-\cos\theta)}\\
&=\frac{174\,\mathrm J}{(9.80\,\mathrm{m/s^2})(0.77495\,\mathrm m)}\\
&=\boxed{22.9\,\mathrm{kg}}.
\end{aligned}$$

**Sense check.** The rise is less than the chain length, as required for an angle below $90^\circ$. The denominator has units $\mathrm{m^2/s^2}$, so dividing joules by it produces kilograms. Substituting $22.9\,\mathrm{kg}$ back into $mg\Delta h$ gives approximately $174\,\mathrm J$.

**Türkçe destek:** $L\cos\theta$ yükselme miktarı değildir; son durumda pivot ile kişi arasındaki dikey uzaklıktır. Yükselmeyi bulmak için başlangıçtaki $L$ uzaklığından bu değeri çıkarıyoruz. Başlangıç ve bitişte hız sıfır olduğu için kinetik enerji terimleri yok olur; aradaki hareketin ayrıntılarını bilmemiz gerekmez.

*Source: 07_testbank.doc, Section 7.2 Problems, Question 2. Original numerical givens retained; restated as an open worked example.*

<a id="qb07-02"></a>

#### Spring energy: convert both centimetre quantities

**Problem.** An athlete stretches an ideal spring by $40.0\,\mathrm{cm}$. Its spring constant is $52.9\,\mathrm{N/cm}$. Find the energy transferred to the spring. Clarified assumption: the spring starts at its unstretched length, as intended by the bank's energy calculation.

**Start from the force that changes with extension.** An ideal spring obeys Hooke's law: its restoring-force magnitude is $kx$. As it stretches, that force grows from zero to $kx$. The work stored as elastic potential energy is therefore the area under the force–extension graph, a triangle rather than a rectangle:

$$U_s=\frac12kx^2.$$

The factor $1/2$ matters because the final force was not acting throughout the whole stretch.

**Convert the spring constant carefully.** One metre contains one hundred centimetres. Thus a force increase of $52.9\,\mathrm N$ per centimetre corresponds to a hundred times as much per metre:

$$\begin{aligned}
k&=52.9\,\frac{\mathrm N}{\mathrm{cm}}
\left(\frac{100\,\mathrm{cm}}{1\,\mathrm m}\right)
=5290\,\mathrm{N/m},\\
x&=40.0\,\mathrm{cm}=0.400\,\mathrm m.
\end{aligned}$$

**Substitute only after conversion.**

$$\begin{aligned}
U_s&=\frac12(5290\,\mathrm{N/m})(0.400\,\mathrm m)^2\\
&=\frac12(5290)(0.160)\,\mathrm{N\,m}\\
&=\boxed{423\,\mathrm J}.
\end{aligned}$$

**Interpret and check.** The final spring-force magnitude is $kx=2116\,\mathrm N$. Its average during this linear stretch is half that, $1058\,\mathrm N$. Multiplying the average by $0.400\,\mathrm m$ gives the same $423.2\,\mathrm J$. Doubling the extension would quadruple the stored energy, because extension is squared.

**Türkçe destek:** $\mathrm{N/cm}$ birimindeki sabiti metreye çevirirken $100$ ile çarparız; paydadaki uzunluk birimi değişmektedir. Uzamayı ise santimetreden metreye çevirirken $100$ ile böleriz. Aynı sayısal dönüşümü ikisine de uygulamak yanlıştır. Enerji hesabında bütün uzamanın karesi alınır.

*Source: 07_testbank.doc, Section 7.2 Problems, Question 3. Adapted clarification: initial spring length is its unstretched length; numerical givens unchanged.*

<a id="qb07-03"></a>

#### Landing on a spring: gravity keeps working during compression

**Problem.** A $60.0\,\mathrm{kg}$ person falls from rest through $1.20\,\mathrm m$ before contacting a massless platform on an ideal massless spring. The platform then moves down $6.00\,\mathrm{cm}$ before the person first comes to rest. Find the spring constant, neglecting losses and using $g=9.80\,\mathrm{m/s^2}$.

**Choose the initial and final states.** Start before the fall, when the person is at rest and the spring is uncompressed. Finish at maximum compression, when the person is momentarily at rest again. Both endpoint kinetic energies are zero. During the complete event gravity lowers the person through the free-fall height plus the extra compression distance.

Let $h=1.20\,\mathrm m$ and $x=0.0600\,\mathrm m$. The total vertical drop is $h+x=1.260\,\mathrm m$, not just $h$. Conservation of mechanical energy gives

$$mg(h+x)=\frac12kx^2.$$

**Isolate the spring constant.** Multiply both sides by $2$, then divide by the square of the compression:

$$\begin{aligned}
2mg(h+x)&=kx^2,\\
k&=\frac{2mg(h+x)}{x^2}\\
&=\frac{2(60.0\,\mathrm{kg})(9.80\,\mathrm{m/s^2})(1.20+0.0600)\,\mathrm m}
{(0.0600\,\mathrm m)^2}\\
&=\boxed{4.12\times10^5\,\mathrm{N/m}}.
\end{aligned}$$

**Interpret the turning point.** Momentarily at rest does not mean force equilibrium. At maximum compression the spring force is $kx=24{,}696\,\mathrm N$, much greater than the person's $588\,\mathrm N$ weight; the resulting upward acceleration begins the rebound. Using $kx=mg$ here would describe static equilibrium instead of this stopping instant.

**Sense check.** The lost gravitational potential energy is $60.0(9.80)(1.260)=740.88\,\mathrm J$. The calculated spring stores $\tfrac12(411600)(0.0600)^2=740.88\,\mathrm J$, so the energy account closes.

**Türkçe destek:** Kişi yayla temas ettikten sonra da aşağı inmeye devam eder. Bu ek $x$ mesafesinde de yerçekimi potansiyel enerjisi azalır; bu yüzden $h+x$ kullanılır. Ayrıca $6.00\,\mathrm{cm}=0.0600\,\mathrm m$ dönüşümü, özellikle $x^2$ nedeniyle sonuca güçlü biçimde etki eder.

*Source: 07_testbank.doc, Section 7.2 Problems, Question 14. Original numerical givens retained; restated as an open worked example.*

<a id="qb07-04"></a>

#### Why dropping a hanging load stretches a spring twice as far

**Problem.** A load is attached to a vertical spring that is initially unstretched. When lowered slowly, it reaches equilibrium after an extension of $6.4\,\mathrm{cm}$. If instead released from rest at the unstretched position, what maximum extension does it reach? Treat the spring as ideal and massless, with negligible dissipative losses.

**Use the slow experiment to identify a relationship.** At static equilibrium the load's weight balances the spring force. If $x_e=6.4\,\mathrm{cm}=0.064\,\mathrm m$ denotes the equilibrium extension,

$$mg=kx_e\quad\Rightarrow\quad\frac{mg}{k}=x_e.$$

We do not need separate numerical values for mass or spring constant; the first experiment provides exactly the ratio needed in the second.

**Now analyse the release by energy.** The load starts at rest with zero extension. At the lowest turning point it is again momentarily at rest, but it has fallen a distance $x$. The loss of gravitational potential energy becomes spring energy:

$$mgx=\frac12kx^2.$$

Move all terms to one side and factor:

$$\begin{aligned}
\frac12kx^2-mgx&=0,\\
x\left(\frac12kx-mg\right)&=0.
\end{aligned}$$

The root $x=0$ is the release position. For the lower turning point use the other factor:

$$\frac12kx-mg=0\quad\Rightarrow\quad x=\frac{2mg}{k}=2x_e.$$

Therefore

$$x_{\max}=2(0.064\,\mathrm m)=\boxed{0.128\,\mathrm m=12.8\,\mathrm{cm}\approx13\,\mathrm{cm}}.$$

**Interpret and check.** At equilibrium the moving load still has kinetic energy and continues downward. At maximum extension the spring force is $kx_{\max}=2mg$, so the net force points upward and reverses the motion. The factor of two is a result of ideal energy conservation, not a general rule for every damped real spring.

**Türkçe destek:** Yavaşça indirme deneyinde kuvvet dengesi, serbest bırakma deneyinde enerji korunumu kullanılır. Aynı denklemi iki duruma da uygulamak doğru değildir. $x$ ile sadeleştirmeden önce $x=0$ kökünün başlangıç konumunu temsil ettiğini görmek, fiziksel olarak aradığımız diğer kökü seçmeyi kolaylaştırır.

*Source: 07_testbank.doc, Section 7.2 Problems, Question 23. Original numerical givens retained; restated as an open worked example.*

<a id="qb07-05"></a>

#### Energy lost during one complete circuit

**Problem.** A $5.00\,\mathrm{kg}$ object travels clockwise around a circle of radius $50.0\,\mathrm{cm}$. Its speed is $4.00\,\mathrm{m/s}$ at one location and $3.00\,\mathrm{m/s}$ when it next returns there. Find the work by dissipative forces. Then find their constant opposing tangential force magnitude.

**Compare the same location at two times.** The object completes one circuit and returns to its original position. Any gravitational or other position-dependent conservative potential energy therefore has the same initial and final value. The work by dissipative forces equals the change in mechanical energy:

$$W_{\rm diss}=\Delta K+\Delta U=\Delta K.$$

The change in kinetic energy must use final minus initial:

$$\begin{aligned}
W_{\rm diss}&=\frac12m(v_f^2-v_i^2)\\
&=\frac12(5.00\,\mathrm{kg})\left[(3.00)^2-(4.00)^2\right]\,\mathrm{m^2/s^2}\\
&=\frac12(5.00)(9.00-16.00)\,\mathrm J\\
&=\boxed{-17.5\,\mathrm J}.
\end{aligned}$$

**Turn the energy loss into a force magnitude.** For a constant tangential resistance opposite the motion, the work is negative force magnitude times path length. The relevant distance is the circle's circumference, not its diameter or the zero net displacement:

$$\ell=2\pi r=2\pi(0.500\,\mathrm m)=\pi\,\mathrm m.$$

Thus

$$W_{\rm diss}=-f\ell\quad\Rightarrow\quad
f=-\frac{W_{\rm diss}}{\ell}
=\frac{17.5\,\mathrm J}{\pi\,\mathrm m}
=\boxed{5.57\,\mathrm N}.$$

**Sense check.** The initial and final kinetic energies are $40.0\,\mathrm J$ and $22.5\,\mathrm J$. Their difference is exactly $17.5\,\mathrm J$, confirming an energy loss rather than a gain. Returning to the same point does not make frictional work zero, because friction depends on the travelled path.

**Türkçe destek:** Yer değiştirme sıfır olsa bile alınan yol $2\pi r$'dir. Sürtünmenin işi bu yol boyunca birikir. $-17.5\,\mathrm J$ işin işaretli değeridir; kuvvetin büyüklüğünü bulurken $f=-W/\ell$ ifadesi pozitif sonuç verir. Yalnızca teğetsel, harekete zıt direnç bileşeni bu iş hesabına girer.

*Source: 07_testbank.doc, Section 7.2 Problems, Question 26. Numerical givens retained; constant dissipative force interpreted as opposing tangential resistance.*

<a id="w08-problems"></a>

## 4. Problem set with step-by-step answers / Problem seti

Try the diagram and the symbolic equation before opening **Answer and steps**. Problems keep their stable *Module XX Pn* identifiers so the complete solution file can be found after its release date.

**TR:** Önce şekli çiz ve denklemi sembolik olarak kur; yanıtı ancak kendi denemenden sonra aç.

### Core problems (L1) / Temel problemler

Everyone should complete these; they follow the worked examples directly.

#### P3 — Gravitational Potential Energy  ·  L1
A $70.0\,\mathrm{kg}$ hiker climbs from an elevation of $1200\,\mathrm m$ to $1850\,\mathrm m$. How much gravitational potential energy does the hiker gain?

**Türkçe problem açıklaması:** $70.0\,\mathrm{kg}$ yürüyüşçü $1200\,\mathrm m$ yükseltiden $1850\,\mathrm m$ yükseltiye çıkıyor. Yerçekimi potansiyel enerjisindeki artışı bulun. Formüldeki yükseklik, son yükselti değil $\Delta h=h_f-h_i$ farkıdır.

<details>
<summary>Answer</summary>

$$\Delta h=1850-1200=650\,\mathrm m,\qquad\Delta U_g=mg\Delta h=(70.0)(9.81)(650)=446355\,\mathrm J\approx446\,\mathrm{kJ}.$$

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 06 P3 in the solutions collection (file `Week_06_Python_Solutions.ipynb`, opens 17 November 2026).

### Intermediate problems (L2) / Orta düzey

Combine two ideas from this week.

#### P5 — Spring Launcher  ·  L2
A spring with $k=800\,\mathrm{N/m}$ is compressed by $0.15\,\mathrm m$ and used to launch a $0.25\,\mathrm{kg}$ ball vertically. (a) What is the stored elastic potential energy? (b) What is the launch speed? (c) How high does the ball rise above the launch point? Ignore friction.

**Türkçe problem açıklaması:** $k=800\,\mathrm{N/m}$ yay $0.15\,\mathrm m\,\mathrm s$ıkıştırılıp $0.25\,\mathrm{kg}$ topu düşey fırlatıyor. (a) Yay enerjisini, (b) ayrılma anındaki sürati, (c) fırlatma noktasından sonraki yükselmeyi bulun. Sürtünme yoktur. “Fırlatma noktası” başlangıçtaki sıkışmış konum mu, yaydan ayrılma konumu mu? Önce bunu belirtin; yay açılırken de yerçekimi enerjisi artar.

<details>
<summary>Answer and assumptions</summary>

(a) The stored spring energy is
$$U_s=\frac12kx^2=\frac12(800)(0.15)^2=9.00\,\mathrm J.$$
For a vertical massless spring, assume the ball separates at the spring's natural length. During expansion it rises $0.15\,\mathrm m$, so gravity takes some of the spring energy:
$$\begin{aligned}K_\mathrm{sep}&=9.00-(0.25)(9.81)(0.15)=8.632125\,\mathrm J,\\
\text{(b)}\quad v_\mathrm{sep}&=\sqrt{\frac{2K_\mathrm{sep}}m}=\sqrt{\frac{2(8.632125)}{0.25}}=8.31\,\mathrm{m/s},\\
\text{(c)}\quad h_\mathrm{after}&=\frac{K_\mathrm{sep}}{mg}=3.52\,\mathrm m,\\
h_\mathrm{total}&=0.15+h_\mathrm{after}=3.67\,\mathrm m.
\end{aligned}$$
Thus the rise is $3.52\,\mathrm m$ above separation or $3.67\,\mathrm m$ above the initially compressed position. “Launch point” must be specified. The simplified values $8.49\,\mathrm{m/s}$ and $3.67\,\mathrm m$ of subsequent rise ignore gravity during spring expansion.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 06 P5 in the solutions collection (file `Week_06_Python_Solutions.ipynb`, opens 17 November 2026).

#### P6 — Incline with Friction (Energy Method)  ·  L2
A $5.0\,\mathrm{kg}$ block starts from rest at the top of a $3.0\,\mathrm m$ high, $30^\circ$ incline. The coefficient of kinetic friction is $\mu_k=0.20$. Find (a) the work done by friction, (b) the speed at the bottom using energy methods, and (c) how far the block slides on the flat frictionless floor before being stopped by a spring ($k=500\,\mathrm{N/m}$).

**Türkçe problem açıklaması:** $5.0\,\mathrm{kg}$ blok, yüksekliği $3.0\,\mathrm m$ ve açısı $30^\circ$ olan eğimde durgunluktan kayıyor; $\mu_k=0.20$. (a) Sürtünmenin işini, (b) alttaki sürati, (c) sürtünmesiz yatay bölümde $k=500\,\mathrm{N/m}$ yayı sıkıştırarak durana kadar hareketi bulun. Eğim boyu $s=h/\sin\theta$ olur. Yayın sıkışması hesaplanabilir; eğim ile yay arasındaki boşluk verilmediği için toplam yatay yol ayrı bir bilinmeyendir.

<details>
<summary>Answer and assumptions</summary>

(a) Friction acts along the ramp, whose length is $s=h/\sin30^\circ=6.00\,\mathrm m$:
$$\begin{aligned}N&=mg\cos30^\circ=42.478546\,\mathrm N,\\
W_f&=-\mu_kNs=-(0.20)(42.478546)(6.00)=-50.974\,\mathrm J.
\end{aligned}$$
(b) Choose zero gravitational energy at the bottom. Starting from rest,
$$\begin{aligned}\frac12mv^2&=mgh+W_f=147.15-50.974255=96.175745\,\mathrm J,\\
v&=\sqrt{\frac{2(96.175745)}{5.0}}=6.202\,\mathrm{m/s}.
\end{aligned}$$
(c) The flat section is frictionless, so the remaining motion energy becomes spring energy:
$$\frac12kx^2=96.175745\,\mathrm J
\quad\Rightarrow\quad x=\sqrt{\frac{2(96.175745)}{500}}=0.6202\,\mathrm m.$$
This is spring compression. If an unspecified gap $D$ separates the incline from the spring, total distance on the flat floor is $D+0.6202\,\mathrm m$; the total cannot be uniquely determined from the statement.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 06 P6 in the solutions collection (file `Week_06_Python_Solutions.ipynb`, opens 17 November 2026).

#### P7 — Roller Coaster Energy  ·  L2
A $500\,\mathrm{kg}$ roller coaster car starts from rest at the top of a $35.0\,\mathrm m$ hill. (a) What is its speed at the bottom? (b) What is its speed at the top of a second hill that is $20.0\,\mathrm m$ high? (c) What is the maximum height of a second hill that the car can clear? Ignore friction.

**Türkçe problem açıklaması:** $500\,\mathrm{kg}$ hız treni $35.0\,\mathrm m$ yükseklikten durgun başlıyor; sürtünme yok. (a) En alttaki sürati, (b) $20.0\,\mathrm m$ yüksekliğindeki ikinci tepedeki sürati, (c) erişilebilecek en büyük yüksekliği bulun. Bir yüksekliğe sıfır süratle ulaşmak ile tepeyi pozitif süratle aşmak farklı koşullardır.

<details>
<summary>Answer and assumptions</summary>

Choose the bottom as zero height. With no losses, cancel $m$ from $mgh_i=\frac12mv^2+mgh_f$:
$$v^2=2g(h_i-h_f),\qquad v=\sqrt{2g(h_i-h_f)}.$$
$$\begin{aligned}\text{(a)}\quad v_\mathrm{bottom}&=\sqrt{2(9.81)(35.0)}=26.2\,\mathrm{m/s},\\
\text{(b)}\quad v_\mathrm{second}&=\sqrt{2(9.81)(35.0-20.0)}=17.2\,\mathrm{m/s}.
\end{aligned}$$
(c) Set $v=0$: the limiting reachable height is $h_f=h_i=35.0\,\mathrm m$. To pass over the summit with positive speed requires $h_f<35.0\,\mathrm m$ in the frictionless particle model.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 06 P7 in the solutions collection (file `Week_06_Python_Solutions.ipynb`, opens 17 November 2026).

### Challenge problems (L3) / İleri düzey

Engineering-style problems with several steps; useful preparation for the exams.

#### P9 — Bungee Jump Analysis  ·  L3
A $65\,\mathrm{kg}$ person jumps from a bridge $45\,\mathrm m$ above a river. The bungee cord has natural length $15\,\mathrm m$ and spring constant $k=85\,\mathrm{N/m}$. (a) How far does the person fall before the cord begins to stretch? (b) Find the maximum extension of the cord (lowest point of the jump) using energy conservation. (c) What is the maximum speed during the fall, and where does it occur? (d) What is the maximum acceleration experienced?

**Türkçe problem açıklaması:** $65\,\mathrm{kg}$ kişi nehirden $45\,\mathrm m$ yüksekteki köprüden atlıyor. İpin doğal boyu $15\,\mathrm m$, yay sabiti $85\,\mathrm{N/m}$. (a) Gerilme başlayana kadar düşüşü, (b) en büyük uzamayı, (c) en büyük sürati ve gerçekleştiği konumu, (d) en büyük net ivmeyi bulun. İp uzaması ile toplam düşüşü ayırın. En büyük sürat, net kuvvetin sıfır olduğu yerde; en büyük ivme ise başka bir yerde oluşur.

<details>
<summary>Answer and assumptions</summary>

(a) The cord first becomes taut after its natural length: $L_0=15\,\mathrm m$.

(b) Let $x$ be extension beyond that natural length. At the lowest point speed is zero, and the total drop is $L_0+x$:
$$\begin{aligned}mg(L_0+x)&=\frac12kx^2,\\
42.5x^2-637.65x-9564.75&=0,\\
x_\mathrm{max}&=\frac{637.65+\sqrt{637.65^2+4(42.5)(9564.75)}}{85}
=24.2746\,\mathrm m.
\end{aligned}$$
Use the positive root. Total fall is $15+24.2746=39.2746\,\mathrm m$, giving $45-39.2746=5.73\,\mathrm m$ clearance for the ideal point-mass model.

(c) While descending, speed increases until the net downward force becomes zero:
$$\begin{aligned}mg-kx_*&=0\quad\Rightarrow\quad x_*=\frac{mg}{k}=7.5018\,\mathrm m,\\
v_\mathrm{max}&=\sqrt{2g(L_0+x_*)-\frac{k}{m}x_*^2}=19.18\,\mathrm{m/s}.
\end{aligned}$$
This occurs $15+7.5018=22.5018\,\mathrm m$ below the bridge.

(d) At maximum extension, the largest net acceleration is upward:
$$a_\mathrm{max}=\frac{kx_\mathrm{max}}m-g=21.93\,\mathrm{m/s^2}=2.236g.$$
Cord force per unit mass is instead $kx_\mathrm{max}/m=3.236g$. Net acceleration and cord force divided by mass are different quantities.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 06 P9 in the solutions collection (file `Week_06_Python_Solutions.ipynb`, opens 17 November 2026).

#### P10 — Regenerative Braking System  ·  L3
An electric vehicle (mass $1800\,\mathrm{kg}$) travels at $100\,\mathrm{km/h}$ and brakes to $30\,\mathrm{km/h}$ while descending a $200\,\mathrm m$ hill with $5^\circ$ slope. The regenerative braking system captures energy with 75% efficiency. (a) What is the change in kinetic energy? (b) What is the change in gravitational PE? (c) How much energy does the regenerative system capture? (d) If the battery stores this energy and later uses it with 90% efficiency for acceleration on flat ground, what final speed can the car reach from $30\,\mathrm{km/h}$ using only the recovered energy?

**Türkçe problem açıklaması:** $1800\,\mathrm{kg}$ elektrikli araç $5^\circ$ eğimli yolda $200\,\mathrm m$ alçalırken $100$'den $30\,\mathrm{km/h}\,\mathrm s$ürate frenliyor. Geri kazanım verimi $75\%$. (a) Kinetik, (b) yerçekimi enerjisi değişimini, (c) depolanan enerjiyi, (d) bu enerjinin $90\%$ verimle düz yolda kullanılmasıyla $30\,\mathrm{km/h}$ başlangıcından erişilen son sürati bulun. $200\,\mathrm m$ eğim boyunca yol olarak alınır; hızları önce SI birimine çevirin ve iki verimi kendi aşamasında uygulayın.

<details>
<summary>Answer and assumptions</summary>

Interpret $200\,\mathrm m$ as distance along the slope. Convert the two speeds before squaring:
$$v_i=\frac{100}{3.6}\,\mathrm{m/s},\qquad v_f=\frac{30}{3.6}\,\mathrm{m/s},\qquad h=200\sin5^\circ=17.4311\,\mathrm m.$$
$$\begin{aligned}\text{(a)}\quad\Delta K&=\frac12(1800)\left[\left(\frac{30}{3.6}\right)^2-\left(\frac{100}{3.6}\right)^2\right]
=-631.944\,\mathrm{kJ},\\
\text{(b)}\quad\Delta U&=-mgh=-(1800)(9.81)(17.4311)=-307.799\,\mathrm{kJ},\\
\text{(c)}\quad E_\mathrm{stored}&=0.75[-(\Delta K+\Delta U)]=704.808\,\mathrm{kJ}.
\end{aligned}$$
The braking stage releases energy from both slowing and descending. For (d), only $90\%$ of the stored energy returns as motion energy:
$$\begin{aligned}\frac12mv_\mathrm{new}^2&=\frac12m\left(\frac{30}{3.6}\right)^2+0.90E_\mathrm{stored},\\
v_\mathrm{new}&=\sqrt{\left(\frac{30}{3.6}\right)^2+\frac{2(0.90)E_\mathrm{stored}}{1800}}
=27.8254\,\mathrm{m/s}=100.17\,\mathrm{km/h}.
\end{aligned}$$
Use joules in the last substitution and retain unrounded intermediate values. A final speed slightly above the original speed is possible because the descent supplied gravitational energy too.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 06 P10 in the solutions collection (file `Week_06_Python_Solutions.ipynb`, opens 17 November 2026).

<a id="w08-exit"></a>

## 5. Exit check / Çıkış kontrolü

Choose a zero height, write initial/final energies, and show where friction belongs without counting gravity twice.

Write one sentence naming the physical principle and one line of algebra you can now explain. **TR:** Sonuca nasıl ulaştığını bir cümleyle anlat; emin olmadığın ilk adımı işaretle.

<details>
<summary>Model responses / Örnek yanıtlar — open after trying</summary>

**Entry:** Net work equals the change in kinetic energy: $W_{\rm net}=K_f-K_i$. An equation relates physical quantities generally; substitution inserts the particular givens with units.

**Exit:** Choose the ramp bottom as zero height. Then $U_f=0$, and $K_i+mgh-f_ks=K_f$. Gravity is already represented by $mgh$, so adding another $+mgh$ as gravitational work would count the same transfer twice. The full numerical ramp solution appears above.

**Visual prediction:** For a vertical launch from zero height with no drag, initially $K_i=\tfrac12mv_0^2$ and $U_i=0$. At the peak $K=0$ and $U=\tfrac12mv_0^2$. On returning to launch height, $U=0$ and kinetic energy returns to its initial value. Their sum remains constant.

</details>

<a id="w08-next"></a>

## Solutions and next week / Çözümler ve gelecek hafta

Complete worked solutions (all subparts, with optional Python) are released on the course page: Module 06 (Work and energy): `Week_06_Python_Solutions.ipynb`, opens 17 November 2026. Until then, use the *Answer and steps* blocks above.

**Next week:** Week 09 — Linear momentum, impulse and collisions / Doğrusal momentum, itme ve çarpışmalar (16 November 2026 – 20 November 2026).

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)